# Ames Housing Price Prediction — Data Understanding & Exploratory Data Analysis

## Project Objective

The objective of this project is to develop an end-to-end machine learning system
for predicting house sale prices using the Ames Housing dataset.

The target variable for this project is `SalePrice`.

This notebook focuses on understanding the dataset before performing
data transformation, feature engineering, and model training.

---

## Objectives of This Notebook

In this stage, we will:

1. Understand the structure of the dataset.
2. Identify numerical and categorical features.
3. Understand the target variable.
4. Analyze missing values.
5. Study feature distributions.
6. Identify potential outliers.
7. Analyze relationships between features and the target.
8. Analyze feature correlations.
9. Identify potential multicollinearity.
10. Translate our observations into machine learning decisions.

> Important:
> This notebook is for data understanding and exploration.
> We will not perform permanent data cleaning, feature engineering,
> model training, or model selection in this stage.

## 1. Import Required Libraries

We begin by importing the libraries required for:

- Data manipulation
- Numerical analysis
- Interactive visualization
- File and path handling

We will use Plotly for interactive visualizations throughout the EDA process.

In [27]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

## 2. Load the Validated Dataset

The dataset used for EDA should come from the output of the Data Ingestion
stage rather than directly from the original raw-data location.

This maintains the separation between pipeline stages:

Raw Dataset
    ↓
Data Ingestion
    ↓
Validated Artifact
    ↓
EDA

Using the ingestion artifact also ensures that our exploratory analysis
is connected to the same data flow that will eventually be used by
the downstream machine learning pipeline.

In [28]:
DATA_PATH = Path(
    "../artifacts/data_ingestion/AmesHousing.txt"
)

df = pd.read_csv(
    DATA_PATH,
    sep="\t"
)
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


### Initial Observation

The dataset has been successfully loaded from the Data Ingestion artifact.

The following sections will establish its dimensions, structure, data types,
and statistical characteristics before we begin deeper analysis.

## 3. Dataset Dimensions

Before analyzing individual variables, we first determine the size of the dataset.

The shape of a DataFrame is represented as:

    (number of rows, number of columns)

Rows represent observations, while columns represent variables/features.

Understanding the dataset dimensions provides the baseline for all subsequent
analysis.

In [29]:
df.shape

(2930, 82)

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style      2930 non

### Interpretation

The dataset contains **2930 observations** and **82 columns**.

Each row represents one housing observation, while the columns contain
characteristics associated with the property and its sale.

The target variable `SalePrice` is included among these columns.

The dataset contains both numerical and categorical information.

At this point, we are only observing the raw structure.
We are not modifying any values.

In [31]:
numerical_columns = (
    df.select_dtypes(
        include=np.number
    ).columns.tolist()
)

categorical_columns = (
    df.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

In [32]:
print(
    "Numerical columns:",
    len(numerical_columns)
)

print(
    "Categorical columns:",
    len(categorical_columns)
)

Numerical columns: 39
Categorical columns: 43


In [33]:
print(numerical_columns)

['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold', 'SalePrice']


In [34]:
print(categorical_columns)

['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC', 'Central Air', 'Electrical', 'Kitchen Qual', 'Functional', 'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature', 'Sale Type', 'Sale Condition']


In [35]:
target = df["SalePrice"]

In [36]:
target.describe()

count      2930.000000
mean     180796.060068
std       79886.692357
min       12789.000000
25%      129500.000000
50%      160000.000000
75%      213500.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [37]:
fig = px.histogram(
    df,
    x="SalePrice",
    nbins=50,
    title="Distribution of SalePrice"
)

fig.show()

In [38]:
target_skewness = df["SalePrice"].skew()

print(
    "SalePrice skewness:",
    target_skewness
)

SalePrice skewness: 1.7435000757376466


In [39]:
fig = px.box(
    df,
    y="SalePrice",
    title="SalePrice Box Plot"
)

fig.show()

### Interpretation

The `SalePrice` distribution shows evidence of right skewness.

This observation is consistent with the presence of relatively expensive
properties that extend the upper tail of the distribution.

We will not transform the target yet. The decision to apply a transformation
will be made later after considering the requirements and assumptions of
the models we evaluate.

In [40]:
missing_summary = (
    df.isna()
      .sum()
      .sort_values(
          ascending=False
      )
)

missing_summary = (
    missing_summary[
        missing_summary > 0
    ]
)

missing_summary

Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Qual        159
Garage Yr Blt      159
Garage Cond        159
Garage Finish      159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Qual           80
Bsmt Cond           80
BsmtFin Type 1      80
Mas Vnr Area        23
Bsmt Full Bath       2
Bsmt Half Bath       2
Total Bsmt SF        1
BsmtFin SF 1         1
BsmtFin SF 2         1
Garage Area          1
Garage Cars          1
Bsmt Unf SF          1
Electrical           1
dtype: int64

In [41]:
missing_percentage = (
    df.isna().mean() * 100
)

missing_percentage = (
    missing_percentage[
        missing_percentage > 0
    ]
    .sort_values(
        ascending=False
    )
)

missing_percentage

Pool QC           99.556314
Misc Feature      96.382253
Alley             93.242321
Fence             80.477816
Mas Vnr Type      60.580205
Fireplace Qu      48.532423
Lot Frontage      16.723549
Garage Qual        5.426621
Garage Cond        5.426621
Garage Yr Blt      5.426621
Garage Finish      5.426621
Garage Type        5.358362
Bsmt Exposure      2.832765
BsmtFin Type 2     2.764505
Bsmt Cond          2.730375
Bsmt Qual          2.730375
BsmtFin Type 1     2.730375
Mas Vnr Area       0.784983
Bsmt Full Bath     0.068259
Bsmt Half Bath     0.068259
BsmtFin SF 1       0.034130
BsmtFin SF 2       0.034130
Electrical         0.034130
Total Bsmt SF      0.034130
Bsmt Unf SF        0.034130
Garage Area        0.034130
Garage Cars        0.034130
dtype: float64

In [42]:
missing_plot = (
    missing_percentage
    .reset_index()
)

missing_plot.columns = [
    "Feature",
    "MissingPercentage"
]

fig = px.bar(
    missing_plot,
    x="MissingPercentage",
    y="Feature",
    orientation="h",
    title="Missing Values by Feature"
)

fig.show()

In [43]:
selected_numeric = [
    "SalePrice",
    "Gr Liv Area",
    "Total Bsmt SF",
    "Garage Area",
    "Year Built",
]
for column in selected_numeric:

    fig = px.histogram(
        df,
        x=column,
        nbins=40,
        title=f"Distribution of {column}"
    )

    fig.show()
    

In [44]:
numerical_features = [
    column
    for column in numerical_columns
    if column != "SalePrice"
]

print(
    "Number of numerical predictor features:",
    len(numerical_features)
)

Number of numerical predictor features: 38


In [45]:
numerical_summary = (
    df[numerical_features]
    .describe()
    .T
)

numerical_summary

,count,mean,std,min,25%,50%,75%,max
Order,2930.0,1.465500e+03,8.459625e+02,1.0,7.332500e+02,1465.5,2.197750e+03,2.930000e+03
PID,2930.0,7.144645e+08,1.887308e+08,526301100.0,5.284770e+08,535453620.0,9.071811e+08,1.007100e+09
MS SubClass,2930.0,5.738737e+01,4.263802e+01,20.0,2.000000e+01,50.0,7.000000e+01,1.900000e+02
Lot Frontage,2440.0,6.922459e+01,2.336533e+01,21.0,5.800000e+01,68.0,8.000000e+01,3.130000e+02
Lot Area,2930.0,1.014792e+04,7.880018e+03,1300.0,7.440250e+03,9436.5,1.155525e+04,2.152450e+05
Overall Qual,2930.0,6.094881e+00,1.411026e+00,1.0,5.000000e+00,6.0,7.000000e+00,1.000000e+01
Overall Cond,2930.0,5.563140e+00,1.111537e+00,1.0,5.000000e+00,5.0,6.000000e+00,9.000000e+00
Year Built,2930.0,1.971356e+03,3.024536e+01,1872.0,1.954000e+03,1973.0,2.001000e+03,2.010000e+03
Year Remod/Add,2930.0,1.984267e+03,2.086029e+01,1950.0,1.965000e+03,1993.0,2.004000e+03,2.010000e+03
Mas Vnr Area,2907.0,1.018968e+02,1.791126e+02,0.0,0.000000e+00,0.0,1.640000e+02,1.600000e+03


In [46]:
numerical_summary["IQR"] = (
    numerical_summary["75%"]
    - numerical_summary["25%"]
)

numerical_summary[
    [
        "mean",
        "50%",
        "std",
        "min",
        "25%",
        "75%",
        "max",
        "IQR",
    ]
]

,mean,50%,std,min,25%,75%,max,IQR
Order,1.465500e+03,1465.5,8.459625e+02,1.0,7.332500e+02,2.197750e+03,2.930000e+03,1.464500e+03
PID,7.144645e+08,535453620.0,1.887308e+08,526301100.0,5.284770e+08,9.071811e+08,1.007100e+09,3.787041e+08
MS SubClass,5.738737e+01,50.0,4.263802e+01,20.0,2.000000e+01,7.000000e+01,1.900000e+02,5.000000e+01
Lot Frontage,6.922459e+01,68.0,2.336533e+01,21.0,5.800000e+01,8.000000e+01,3.130000e+02,2.200000e+01
Lot Area,1.014792e+04,9436.5,7.880018e+03,1300.0,7.440250e+03,1.155525e+04,2.152450e+05,4.115000e+03
Overall Qual,6.094881e+00,6.0,1.411026e+00,1.0,5.000000e+00,7.000000e+00,1.000000e+01,2.000000e+00
Overall Cond,5.563140e+00,5.0,1.111537e+00,1.0,5.000000e+00,6.000000e+00,9.000000e+00,1.000000e+00
Year Built,1.971356e+03,1973.0,3.024536e+01,1872.0,1.954000e+03,2.001000e+03,2.010000e+03,4.700000e+01
Year Remod/Add,1.984267e+03,1993.0,2.086029e+01,1950.0,1.965000e+03,2.004000e+03,2.010000e+03,3.900000e+01
Mas Vnr Area,1.018968e+02,0.0,1.791126e+02,0.0,0.000000e+00,1.640000e+02,1.600000e+03,1.640000e+02


In [47]:
for column in numerical_features:

    fig = px.histogram(
        df,
        x=column,
        nbins=40,
        title=f"Distribution of {column}"
    )

    fig.show()

In [48]:
for column in numerical_features:

    fig = px.box(
        df,
        y=column,
        title=f"Box Plot of {column}"
    )

    fig.show()

In [49]:
outlier_summary = []

for column in numerical_features:

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (df[column] < lower_bound)
        | (df[column] > upper_bound)
    ).sum()

    outlier_percentage = (
        outlier_count / len(df) * 100
    )

    outlier_summary.append(
        {
            "Feature": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "LowerBound": lower_bound,
            "UpperBound": upper_bound,
            "OutlierCount": outlier_count,
            "OutlierPercentage": outlier_percentage,
        }
    )

outlier_summary = pd.DataFrame(
    outlier_summary
)

outlier_summary.sort_values(
    "OutlierPercentage",
    ascending=False
)

,Feature,Q1,Q3,IQR,LowerBound,UpperBound,OutlierCount,OutlierPercentage
31,Enclosed Porch,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,459,15.665529
11,BsmtFin SF 2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,351,11.979522
33,Screen Porch,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,256,8.737201
6,Overall Cond,5.000000e+00,6.000000e+00,1.000000e+00,3.500000e+00,7.500000e+00,252,8.600683
2,MS SubClass,2.000000e+01,7.000000e+01,5.000000e+01,-5.500000e+01,1.450000e+02,208,7.098976
9,Mas Vnr Area,0.000000e+00,1.640000e+02,1.640000e+02,-2.460000e+02,4.100000e+02,200,6.825939
3,Lot Frontage,5.800000e+01,8.000000e+01,2.200000e+01,2.500000e+01,1.130000e+02,187,6.382253
19,Bsmt Half Bath,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,175,5.972696
30,Open Porch SF,0.000000e+00,7.000000e+01,7.000000e+01,-1.050000e+02,1.750000e+02,159,5.426621
23,Kitchen AbvGr,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,134,4.573379


In [51]:
df['Gr Liv Area'].describe()

count    2930.000000
mean     1499.690444
std       505.508887
min       334.000000
25%      1126.000000
50%      1442.000000
75%      1742.750000
max      5642.000000
Name: Gr Liv Area, dtype: float64

In [53]:
feature = "Gr Liv Area"

fig = px.scatter(
    df,
    x=feature,
    y="SalePrice",
    title=f"{feature} vs SalePrice",
    trendline="ols"
)

fig.show()

### Interpretation

`Gr Liv Area` shows a positive relationship with `SalePrice`.

As above-ground living area increases, sale price generally tends to
increase.

However, the observations do not fall exactly on a straight line.
There is considerable variation in sale price for houses with similar
living areas.

This indicates that living area alone cannot completely explain
house prices.

Other characteristics such as quality, location, age, garage,
basement, and other property attributes may also contribute to
the final sale price.

In [54]:
covariance = df[
    ["Gr Liv Area", "SalePrice"]
].cov()

covariance

,Gr Liv Area,SalePrice
Gr Liv Area,2.555392e+05,2.854220e+07
SalePrice,2.854220e+07,6.381884e+09


In [55]:
numerical_target_correlation = (
    df[numerical_features + ["SalePrice"]]
    .corr()["SalePrice"]
    .drop("SalePrice")
    .sort_values(
        ascending=False
    )
)

numerical_target_correlation

Overall Qual       0.799262
Gr Liv Area        0.706780
Garage Cars        0.647877
Garage Area        0.640401
Total Bsmt SF      0.632280
1st Flr SF         0.621676
Year Built         0.558426
Full Bath          0.545604
Year Remod/Add     0.532974
Garage Yr Blt      0.526965
Mas Vnr Area       0.508285
TotRms AbvGrd      0.495474
Fireplaces         0.474558
BsmtFin SF 1       0.432914
Lot Frontage       0.357318
Wood Deck SF       0.327143
Open Porch SF      0.312951
Half Bath          0.285056
Bsmt Full Bath     0.276050
2nd Flr SF         0.269373
Lot Area           0.266549
Bsmt Unf SF        0.182855
Bedroom AbvGr      0.143913
Screen Porch       0.112151
Pool Area          0.068403
Mo Sold            0.035259
3Ssn Porch         0.032225
BsmtFin SF 2       0.005891
Misc Val          -0.015691
Yr Sold           -0.030569
Order             -0.031408
Bsmt Half Bath    -0.035835
Low Qual Fin SF   -0.037660
MS SubClass       -0.085092
Overall Cond      -0.101697
Kitchen AbvGr     -0

In [56]:
correlation_plot = (
    numerical_target_correlation
    .reset_index()
)

correlation_plot.columns = [
    "Feature",
    "Correlation"
]

fig = px.bar(
    correlation_plot,
    x="Correlation",
    y="Feature",
    orientation="h",
    title="Numerical Feature Correlation with SalePrice"
)

fig.show()

In [57]:
top_correlations = (
    numerical_target_correlation
    .abs()
    .sort_values(
        ascending=False
    )
    .head(10)
)

top_correlations

Overall Qual      0.799262
Gr Liv Area       0.706780
Garage Cars       0.647877
Garage Area       0.640401
Total Bsmt SF     0.632280
1st Flr SF        0.621676
Year Built        0.558426
Full Bath         0.545604
Year Remod/Add    0.532974
Garage Yr Blt     0.526965
Name: SalePrice, dtype: float64

In [58]:
top_features = (
    top_correlations.index
)

df[
    list(top_features) + ["SalePrice"]
].corr()["SalePrice"].sort_values(
    ascending=False
)

SalePrice         1.000000
Overall Qual      0.799262
Gr Liv Area       0.706780
Garage Cars       0.647877
Garage Area       0.640401
Total Bsmt SF     0.632280
1st Flr SF        0.621676
Year Built        0.558426
Full Bath         0.545604
Year Remod/Add    0.532974
Garage Yr Blt     0.526965
Name: SalePrice, dtype: float64

In [59]:
for feature in top_features:

    fig = px.scatter(
        df,
        x=feature,
        y="SalePrice",
        title=f"{feature} vs SalePrice",
        trendline="ols"
    )

    fig.show()

In [61]:
covariance = df[
    ["Gr Liv Area", "SalePrice"]
].cov()

covariance

,Gr Liv Area,SalePrice
Gr Liv Area,2.555392e+05,2.854220e+07
SalePrice,2.854220e+07,6.381884e+09


In [64]:
feature_correlation = (
    df[numerical_features]
    .corr()
)

feature_correlation

,Order,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,...,Garage Area,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
Order,1.000000,0.173593,0.011797,-0.007034,0.031354,-0.048500,-0.011054,-0.052319,-0.075566,-0.030907,...,-0.035435,-0.011292,0.016355,0.027908,-0.024975,0.004307,0.052518,-0.006083,0.133365,-0.975993
PID,0.173593,1.000000,-0.001281,-0.096918,0.034868,-0.263147,0.104451,-0.343388,-0.157111,-0.229283,...,-0.210606,-0.051135,-0.071311,0.162519,-0.024894,-0.025735,-0.002845,-0.008260,-0.050455,0.009579
MS SubClass,0.011797,-0.001281,1.000000,-0.420135,-0.204613,0.039419,-0.067349,0.036579,0.043397,0.002730,...,-0.103239,-0.017310,-0.014823,-0.022866,-0.037956,-0.050614,-0.003434,-0.029254,0.000350,-0.017905
Lot Frontage,-0.007034,-0.096918,-0.420135,1.000000,0.491313,0.212042,-0.074448,0.121562,0.091712,0.222407,...,0.358505,0.120084,0.163040,0.012758,0.028564,0.076666,0.173947,0.044476,0.011085,-0.007547
Lot Area,0.031354,0.034868,-0.204613,0.491313,1.000000,0.097188,-0.034759,0.023258,0.021682,0.126830,...,0.212822,0.157212,0.103760,0.021868,0.016243,0.055044,0.093775,0.069188,0.003859,-0.023085
Overall Qual,-0.048500,-0.263147,0.039419,0.212042,0.097188,1.000000,-0.094812,0.597027,0.569609,0.429418,...,0.563503,0.255663,0.298412,-0.140332,0.018240,0.041615,0.030399,0.005179,0.031103,-0.020719
Overall Cond,-0.011054,0.104451,-0.067349,-0.074448,-0.034759,-0.094812,1.000000,-0.368773,0.047680,-0.135340,...,-0.153754,0.020344,-0.068934,0.071459,0.043852,0.044055,-0.016787,0.034056,-0.007295,0.031207
Year Built,-0.052319,-0.343388,0.036579,0.121562,0.023258,0.597027,-0.368773,1.000000,0.612095,0.313292,...,0.480131,0.228964,0.198365,-0.374364,0.015803,-0.041436,0.002213,-0.011011,0.014577,-0.013197
Year Remod/Add,-0.075566,-0.157111,0.043397,0.091712,0.021682,0.569609,0.047680,0.612095,1.000000,0.196928,...,0.376438,0.217857,0.241748,-0.220383,0.037412,-0.046888,-0.011410,-0.003132,0.018048,0.032652
Mas Vnr Area,-0.030907,-0.229283,0.002730,0.222407,0.126830,0.429418,-0.135340,0.313292,0.196928,1.000000,...,0.373458,0.165467,0.143748,-0.110787,0.013778,0.065643,0.004617,0.044934,-0.000276,-0.017715


In [65]:
fig = px.imshow(
    feature_correlation,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Matrix — Numerical Features"
)

fig.show()

In [66]:
upper_triangle = (
    feature_correlation.where(
        np.triu(
            np.ones(
                feature_correlation.shape
            ),
            k=1
        ).astype(bool)
    )
)

high_correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlation_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_correlation_pairs[
    high_correlation_pairs["Correlation"].abs() >= 0.80
].sort_values(
    "Correlation",
    key=lambda x: x.abs(),
    ascending=False
)

,Feature_1,Feature_2,Correlation
37,Order,Yr Sold,-0.975993
1054,Garage Cars,Garage Area,0.889676
292,Year Built,Garage Yr Blt,0.834849
670,Gr Liv Area,TotRms AbvGrd,0.807772
508,Total Bsmt SF,1st Flr SF,0.800720


In [67]:
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

In [68]:
vif_data = df[numerical_features].copy()

vif_data = vif_data.dropna()

vif_results = pd.DataFrame()

vif_results["Feature"] = vif_data.columns

vif_results["VIF"] = [
    variance_inflation_factor(
        vif_data.values,
        i
    )
    for i in range(vif_data.shape[1])
]

vif_results.sort_values(
    "VIF",
    ascending=False
)

e:\Project_Works\ML_Learn\Ames Housing Prices\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Feature,VIF
16,Low Qual Fin SF,inf
17,Gr Liv Area,inf
10,BsmtFin SF 1,inf
11,BsmtFin SF 2,inf
12,Bsmt Unf SF,inf
13,Total Bsmt SF,inf
15,2nd Flr SF,inf
14,1st Flr SF,inf
37,Yr Sold,2.441114e+04
1,PID,1.926718e+01


In [69]:
vif_plot = (
    vif_results
    .sort_values(
        "VIF",
        ascending=True
    )
)

fig = px.bar(
    vif_plot,
    x="VIF",
    y="Feature",
    orientation="h",
    title="Variance Inflation Factor — Numerical Features"
)

fig.show()

In [70]:
categorical_summary = pd.DataFrame({
    "Feature": categorical_columns,
    "UniqueCategories": [
        df[column].nunique(
            dropna=False
        )
        for column in categorical_columns
    ],
    "MissingValues": [
        df[column].isna().sum()
        for column in categorical_columns
    ]
})

categorical_summary.sort_values(
    "UniqueCategories",
    ascending=False
)

,Feature,UniqueCategories,MissingValues
8,Neighborhood,28,0
16,Exterior 2nd,17,0
15,Exterior 1st,16,0
41,Sale Type,10,0
9,Condition 1,9,0
12,House Style,8,0
31,Functional,8,0
14,Roof Matl,8,0
10,Condition 2,8,0
0,MS Zoning,7,0


In [71]:
neighborhood_counts = (
    df[feature]
    .value_counts(
        dropna=False
    )
)

neighborhood_counts

Garage Yr Blt
NaN       159
2005.0    142
2006.0    115
2007.0    115
2004.0     99
         ... 
1895.0      1
1933.0      1
2207.0      1
1943.0      1
1919.0      1
Name: count, Length: 104, dtype: int64

In [72]:
fig = px.bar(
    neighborhood_counts,
    title="Neighborhood Frequency"
)

fig.show()

In [73]:
neighborhood_price_summary = (
    df.groupby("Neighborhood")["SalePrice"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
        ]
    )
    .sort_values(
        "median",
        ascending=False
    )
)

neighborhood_price_summary

,count,mean,median,std,min,max
Neighborhood,,,,,,
StoneBr,51,324229.196078,319000.0,119273.020694,130000,591587
NridgHt,166,322018.265060,317750.0,95932.354274,154000,615000
NoRidge,71,330319.126761,302000.0,101444.662927,190000,755000
GrnHill,2,280000.000000,280000.0,70710.678119,230000,330000
Veenker,24,248314.583333,250250.0,65474.949540,150000,385000
Timber,72,246599.541667,232106.5,69326.471547,137500,425000
Somerst,182,229707.324176,225500.0,57437.392588,139000,468000
Crawfor,103,207550.834951,200624.0,65230.183286,90350,392500
CollgCr,267,201803.434457,200000.0,54187.843749,110000,475000


In [74]:
fig = px.box(
    df,
    x="Neighborhood",
    y="SalePrice",
    title="SalePrice Distribution by Neighborhood"
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()

In [75]:
categorical_overview = []

for column in categorical_columns:

    categorical_overview.append(
        {
            "Feature": column,
            "UniqueCategories": df[column].nunique(
                dropna=False
            ),
            "MissingValues": df[column].isna().sum(),
            "MostFrequent": df[column].mode(
                dropna=True
            ).iloc[0]
            if not df[column].mode(
                dropna=True
            ).empty
            else np.nan,
            "MostFrequentCount": df[column].value_counts(
                dropna=False
            ).iloc[0],
        }
    )

categorical_overview = pd.DataFrame(
    categorical_overview
)

categorical_overview

,Feature,UniqueCategories,MissingValues,MostFrequent,MostFrequentCount
0,MS Zoning,7,0,RL,2273
1,Street,2,0,Pave,2918
2,Alley,3,2732,Grvl,2732
3,Lot Shape,4,0,Reg,1859
4,Land Contour,4,0,Lvl,2633
5,Utilities,3,0,AllPub,2927
6,Lot Config,5,0,Inside,2140
7,Land Slope,3,0,Gtl,2789
8,Neighborhood,28,0,NAmes,443
9,Condition 1,9,0,Norm,2522


In [76]:
missing_profile = pd.DataFrame({
    "Feature": df.columns,
    "MissingCount": [
        df[column].isna().sum()
        for column in df.columns
    ],
    "MissingPercentage": [
        df[column].isna().mean() * 100
        for column in df.columns
    ],
    "DataType": [
        df[column].dtype
        for column in df.columns
    ],
    "UniqueValues": [
        df[column].nunique(dropna=True)
        for column in df.columns
    ]
})

missing_profile = (
    missing_profile[
        missing_profile["MissingCount"] > 0
    ]
    .sort_values(
        "MissingPercentage",
        ascending=False
    )
)

missing_profile

,Feature,MissingCount,MissingPercentage,DataType,UniqueValues
73,Pool QC,2917,99.556314,str,4
75,Misc Feature,2824,96.382253,str,5
7,Alley,2732,93.242321,str,2
74,Fence,2358,80.477816,str,4
26,Mas Vnr Type,1775,60.580205,str,4
58,Fireplace Qu,1422,48.532423,str,5
4,Lot Frontage,490,16.723549,float64,128
64,Garage Qual,159,5.426621,str,5
65,Garage Cond,159,5.426621,str,5
60,Garage Yr Blt,159,5.426621,float64,103


In [77]:
fig = px.bar(
    missing_profile.sort_values(
        "MissingPercentage"
    ),
    x="MissingPercentage",
    y="Feature",
    orientation="h",
    title="Missing Value Percentage by Feature"
)

fig.show()

In [78]:
analysis_df = df[
    ["Garage Type", "SalePrice"]
].copy()

analysis_df["GarageType_Missing"] = (
    analysis_df["Garage Type"].isna()
)

analysis_df.groupby(
    "GarageType_Missing"
)["SalePrice"].agg(
    [
        "count",
        "mean",
        "median",
        "std"
    ]
)

,count,mean,median,std
GarageType_Missing,,,,
False,2773,185090.307609,165000.0,79584.836009
True,157,104949.254777,100000.0,34069.812774


In [79]:
fig = px.box(
    analysis_df,
    x="GarageType_Missing",
    y="SalePrice",
    title="SalePrice by Garage Type Missingness"
)

fig.show()

In [80]:
feature_profile = pd.DataFrame({
    "Feature": df.columns,
    "DataType": [
        str(df[column].dtype)
        for column in df.columns
    ],
    "UniqueValues": [
        df[column].nunique(dropna=True)
        for column in df.columns
    ],
    "MissingCount": [
        df[column].isna().sum()
        for column in df.columns
    ],
    "MissingPercentage": [
        df[column].isna().mean() * 100
        for column in df.columns
    ],
})

In [81]:
feature_profile["Mean"] = np.nan
feature_profile["Median"] = np.nan
feature_profile["Std"] = np.nan
feature_profile["Min"] = np.nan
feature_profile["Max"] = np.nan
feature_profile["Skewness"] = np.nan

In [82]:
for column in numerical_features:

    mask = (
        feature_profile["Feature"]
        == column
    )

    feature_profile.loc[
        mask,
        "Mean"
    ] = df[column].mean()

    feature_profile.loc[
        mask,
        "Median"
    ] = df[column].median()

    feature_profile.loc[
        mask,
        "Std"
    ] = df[column].std()

    feature_profile.loc[
        mask,
        "Min"
    ] = df[column].min()

    feature_profile.loc[
        mask,
        "Max"
    ] = df[column].max()

    feature_profile.loc[
        mask,
        "Skewness"
    ] = df[column].skew()